# First Pass at Cleaning Lyft Data

Each CSV holds about one month of ridership data.

## Mount drive to access 255 Final Project folder

In [ ]:
import os

BASE_DIR = os.path.dirname(os.path.abspath(_file_)) if '_file_' in dir() else os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

## Requirements

In [ ]:
import pandas as pd
import geopandas as gpd

# Data Cleaning Functions

In [ ]:
###
# This function will take a Lyft ridership data CSV (which holds ~1 month of
# ridership data) and return a dataframe with only rides that start, end, or both
# in Oakland or Berkeley.
#
# Parameters:
#   csv_file: string -> path to csv with ridership data
#
# Returns:
#   dataframe -> holds only Oakland/Berkeley rows of the input csv
###
def create_oak_berk_df(csv_file):
  # Read in csv and save as dataframe
  df = pd.read_csv(csv_file)
  # Create mask to filter only rides that either start, end, or both in Berkeley/Oakland
  oak_berk_start_mask = df['start_station_id'].str.startswith('BK') | df['start_station_id'].str.startswith('OK')
  oak_berk_end_mask = df['end_station_id'].str.startswith('BK') | df['end_station_id'].str.startswith('OK')
  oak_berk_mask = oak_berk_start_mask | oak_berk_end_mask
  # Apply mask to create dataframe with only rides in Berkeley/Oakland
  oak_berk_df = df.loc[oak_berk_mask]
  # Return filtered dataframe
  return oak_berk_df


In [ ]:
# This function will take a ridership dataframe and return the station counts
# for where rides start, end, and a combined count.
#
# Parameters:
#   df: pandas.Dataframe -> aggregate dataframe holding all data for
#       Oakland/Berkeley rides
#
# Returns:
#   stations_gdf: (pandas.GeoDataFrame) ->
#      id column holds station id
#      start_count column holds number of rides that start there
#      end_count holds number of rides that end there
#      combined_count is the sum of start_count and end_count
#      name
#      lat
#      long
#      geometry
#
def create_station_counts_gdf(df):
  # Filter only non-SF, non-SJ start stations to get value counts for starting stations
  # and filter only non-SF, non-SJ end stations to get value counts for ending stations.
  # Drop row if starting id or ending id is NaN.
  start_df = df.loc[
      df['start_station_id'].notna() &
      ~df['start_station_id'].fillna('').str.startswith("SF") &
      ~df['start_station_id'].fillna('').str.startswith("SJ")
  ]
  end_df = df.loc[
      df['end_station_id'].notna() &
      ~df['end_station_id'].fillna('').str.startswith("SF") &
      ~df['start_station_id'].fillna('').str.startswith("SJ")
  ]

  # Get value counts and combine into single dataframe
  start_counts = start_df['start_station_id'].value_counts()
  end_counts = end_df['end_station_id'].value_counts()
  station_counts = pd.DataFrame({
      "start_count": start_counts,
      "end_count": end_counts
  })
  station_counts = station_counts.reset_index().rename(columns={"index": "id"})
  # Fill in zero for missing values
  station_counts = station_counts.fillna(0)
  # Change from float to int
  station_counts["start_count"] = station_counts["start_count"].astype(int)
  station_counts["end_count"] = station_counts["end_count"].astype(int)

  # Add combined count column
  station_counts['combined_count'] = station_counts['start_count'] + station_counts['end_count']

  # Get station names
  station_names = pd.concat([
      df[["start_station_id", "start_station_name"]].rename(
          columns={"start_station_id": "id", "start_station_name": "name"}
      ),
      df[["end_station_id", "end_station_name"]].rename(
          columns={"end_station_id": "id", "end_station_name": "name"}
      )
  ]).dropna(subset=["id"]).drop_duplicates()
  print("station names length: ", len(station_names))


  # Get station locations
  station_locations = pd.concat([
      df[["start_station_id", "start_lat", "start_lng"]].rename(
          columns={"start_station_id": "id", "start_lat": "lat", "start_lng": "long"}
      ),
      df[["end_station_id", "end_lat", "end_lng"]].rename(
          columns={"end_station_id": "id", "end_lat": "lat", "end_lng": "long"}
      )
  ]).dropna(subset=["id"]).drop_duplicates()
  print("station locations length: ", len(station_locations))

  # Take first name and coordinates for each station id in case of conflicting info
  station_names = station_names.groupby("id", as_index=False).first()
  station_locations = station_locations.groupby("id", as_index=False).first()

  # Add station name and location columns to dataframe
  station_counts = (
      station_counts
        .merge(station_names, on="id", how="left")
        .merge(station_locations, on="id", how="left")
  )

  # Turn into geodataframe with crs 4326
  station_counts_gdf = gpd.GeoDataFrame(
    station_counts,
    geometry=gpd.points_from_xy(station_counts["long"], station_counts["lat"]),
    crs="EPSG:4326"
  )

  return station_counts_gdf

In [ ]:
i = 14
f"{(i+1):02}"

In [ ]:
def read_in_data(year_str):
  fpath_prefix = os.path.join(DATA_DIR, 'raw/BikeshareData/')
  fpath_suffix = '-baywheels-tripdata.csv'
  oak_berk_df_list = []

  for i in range(12):
    if (year_str == '2019' and i < 4):
      # Lyft data starts from 5th month of 2019
      continue
    elif (year_str == '2020' and i < 4):
      # station ids with city initial starts from 5th month of 2020
      continue
    elif (year_str == '2020' and i == 3):
      # Randomly Lyft is missing 4th month of 2020 data
      continue
    elif (year_str == '2026' and i > 1):
      # Only first few months of 2026 data exist
      break
    # Get file path
    fpath_str = year_str + f"{(i+1):02}"
    file_path = fpath_prefix + fpath_str + fpath_suffix
    print(file_path)

    # Create oakland berkeley dataframe and add to list
    df = create_oak_berk_df(file_path)
    oak_berk_df_list.append(df)

  # Concat all dataframes into one dataframe and return it
  oak_berk_df = pd.concat(oak_berk_df_list)
  return oak_berk_df

# Clean data and export to CSV

In [ ]:
oak_berk_df_2019 = read_in_data('2019')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/201905-baywheels-tripdata.csv


AttributeError: Can only use .str accessor with string values!

In [ ]:
oak_berk_df_2020 = read_in_data('2020')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202005-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202006-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202007-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202008-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202009-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202010-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202011-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202012-baywheels-tripdata.csv


In [ ]:
oak_berk_df_2021 = read_in_data('2021')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202101-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202102-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202103-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202104-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202105-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202106-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202107-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202108-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202109-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202110-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202111-baywheels-tripdata.csv
/content/d

In [ ]:
oak_berk_df_2022 = read_in_data('2022')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202201-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202202-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202203-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202204-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202205-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202206-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202207-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202208-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202209-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202210-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202211-baywheels-tripdata.csv
/content/d

In [ ]:
oak_berk_df_2023 = read_in_data('2023')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202301-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202302-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202303-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202304-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202305-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202306-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202307-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202308-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202309-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202310-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202311-baywheels-tripdata.csv
/content/d

In [ ]:
oak_berk_df_2024 = read_in_data('2024')

/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202401-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202402-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202403-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202404-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202405-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202406-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202407-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202408-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202409-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202410-baywheels-tripdata.csv
/content/drive/MyDrive/255 Final Project/Data/BikeshareData/202411-baywheels-tripdata.csv
/content/d

In [ ]:
oak_berk_df_2025 = read_in_data('2025')

In [ ]:
oak_berk_df_2026 = read_in_data('2026')

In [ ]:
oak_berk_df_2026

In [ ]:
# check station ids are consistently unique with name
pd.set_option('display.max_rows', None)
oak_berk_df_2026.groupby("start_station_name")[["start_station_id"]].nunique()


In [ ]:
oak_berk_df_2026.loc[oak_berk_df_2026['start_station_id'].isna()]

In [ ]:
oak_berk_df_2020.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/OaklandBerkeleyData2020.csv'), index=False)

In [ ]:
oak_berk_df_2021.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/OaklandBerkeleyData2021.csv'), index=False)

In [ ]:
oak_berk_df_2022.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/OaklandBerkeleyData2022.csv'), index=False)

In [ ]:
oak_berk_df_2023.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/aklandBerkeleyData2023.csv'), index=False)

In [ ]:
oak_berk_df_2024.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/OaklandBerkeleyData2024.csv'), index=False)

In [ ]:
oak_berk_df_2025.to_csv(os.path.join(DATA_DIR, 'cleaned/rides by year/OaklandBerkeleyData2025.csv'), index=False)

In [ ]:
# oak_berk_df_2026.to_csv(os.path.join(DATA_DIR, 'OaklandBerkeleyData2026.csv'), index=False)

# Create cleaned geodataframes and export

In [ ]:
counts_gdf_20 = create_station_counts_gdf(oak_berk_df_2020)
counts_gdf_20

station names length:  176
station locations length:  389


,id,start_count,end_count,combined_count,name,lat,long,geometry
0,16th St Depot,0,3,3,16th St Depot,37.766349,-122.396292,POINT (-122.39629 37.76635)
1,BK-A3,708,700,1408,College Ave at Alcatraz Ave,37.851376,-122.252523,POINT (-122.25252 37.85138)
2,BK-A7,1095,980,2075,Vine St at Shattuck Ave,37.880222,-122.269592,POINT (-122.26959 37.88022)
3,BK-B7,569,407,976,Virginia St at Shattuck Ave,37.876572,-122.269527,POINT (-122.26953 37.87657)
4,BK-B9,1192,551,1743,Hearst Ave at Euclid Ave,37.875112,-122.260553,POINT (-122.26055 37.87511)
...,...,...,...,...,...,...,...,...
126,OK-M6,729,691,1420,Franklin St at 9th St,37.800516,-122.272080,POINT (-122.27208 37.80052)
127,OK-M7,1228,981,2209,Lake Merritt BART Station,37.797320,-122.265320,POINT (-122.26532 37.79732)
128,OK-N17,220,213,433,Fruitvale BART Station,37.775232,-122.224498,POINT (-122.2245 37.77523)
129,OK-N6,867,981,1848,Jack London Square,37.796248,-122.279352,POINT (-122.27935 37.79625)


In [ ]:
df_20 = counts_gdf_20.drop(columns='geometry')
df_20

,id,start_count,end_count,combined_count,name,lat,long
0,16th St Depot,0,3,3,16th St Depot,37.766349,-122.396292
1,BK-A3,708,700,1408,College Ave at Alcatraz Ave,37.851376,-122.252523
2,BK-A7,1095,980,2075,Vine St at Shattuck Ave,37.880222,-122.269592
3,BK-B7,569,407,976,Virginia St at Shattuck Ave,37.876572,-122.269527
4,BK-B9,1192,551,1743,Hearst Ave at Euclid Ave,37.875112,-122.260553
...,...,...,...,...,...,...,...
126,OK-M6,729,691,1420,Franklin St at 9th St,37.800516,-122.272080
127,OK-M7,1228,981,2209,Lake Merritt BART Station,37.797320,-122.265320
128,OK-N17,220,213,433,Fruitvale BART Station,37.775232,-122.224498
129,OK-N6,867,981,1848,Jack London Square,37.796248,-122.279352


In [ ]:
counts_gdf_21 = create_station_counts_gdf(oak_berk_df_2021)
counts_gdf_21

station names length:  165
station locations length:  315


,id,start_count,end_count,combined_count,name,lat,long,geometry
0,BK-A3,1167,971,2138,College Ave at Alcatraz Ave,37.851376,-122.252523,POINT (-122.25252 37.85138)
1,BK-A7,1424,1312,2736,Vine St at Shattuck Ave,37.880222,-122.269592,POINT (-122.26959 37.88022)
2,BK-B7,1267,928,2195,Virginia St at Shattuck Ave,37.876573,-122.269528,POINT (-122.26953 37.87657)
3,BK-B9,72,49,121,Hearst Ave at Euclid Ave,37.875112,-122.260553,POINT (-122.26055 37.87511)
4,BK-C1,724,979,1703,Fifth St at Delaware St,37.870407,-122.299676,POINT (-122.29968 37.87041)
...,...,...,...,...,...,...,...,...
135,OK-N1,1948,2024,3972,West Oakland BART Station,37.805318,-122.294837,POINT (-122.29484 37.80532)
136,OK-N17,208,200,408,Fruitvale BART Station,37.775232,-122.224498,POINT (-122.2245 37.77523)
137,OK-N5,744,918,1662,Jack London Square,37.796248,-122.279352,POINT (-122.27935 37.79625)
138,OK-N6,927,1090,2017,Webster St at 2nd St,37.795195,-122.273970,POINT (-122.27397 37.79519)


In [ ]:
df_21 = counts_gdf_21.drop(columns='geometry')
df_21

,id,start_count,end_count,combined_count,name,lat,long
0,BK-A3,1167,971,2138,College Ave at Alcatraz Ave,37.851376,-122.252523
1,BK-A7,1424,1312,2736,Vine St at Shattuck Ave,37.880222,-122.269592
2,BK-B7,1267,928,2195,Virginia St at Shattuck Ave,37.876573,-122.269528
3,BK-B9,72,49,121,Hearst Ave at Euclid Ave,37.875112,-122.260553
4,BK-C1,724,979,1703,Fifth St at Delaware St,37.870407,-122.299676
...,...,...,...,...,...,...,...
135,OK-N1,1948,2024,3972,West Oakland BART Station,37.805318,-122.294837
136,OK-N17,208,200,408,Fruitvale BART Station,37.775232,-122.224498
137,OK-N5,744,918,1662,Jack London Square,37.796248,-122.279352
138,OK-N6,927,1090,2017,Webster St at 2nd St,37.795195,-122.273970


In [ ]:
counts_gdf_22 = create_station_counts_gdf(oak_berk_df_2022)
counts_gdf_22

station names length:  166
station locations length:  287


,id,start_count,end_count,combined_count,name,lat,long,geometry
0,BK-A3,1170,1031,2201,College Ave at Alcatraz Ave,37.851376,-122.252523,POINT (-122.25252 37.85138)
1,BK-A7,1527,1459,2986,Vine St at Shattuck Ave,37.880222,-122.269592,POINT (-122.26959 37.88022)
2,BK-B7,826,736,1562,Virginia St at Shattuck Ave,37.876573,-122.269528,POINT (-122.26953 37.87657)
3,BK-C1,966,1310,2276,Fifth St at Delaware St,37.870407,-122.299676,POINT (-122.29968 37.87041)
4,BK-C5,1460,1504,2964,North Berkeley BART Station,37.873558,-122.283093,POINT (-122.28309 37.87356)
...,...,...,...,...,...,...,...,...
127,OK-M7,2793,2284,5077,Lake Merritt BART Station,37.797320,-122.265320,POINT (-122.26532 37.79732)
128,OK-N1,2981,2987,5968,West Oakland BART Station,37.805189,-122.294617,POINT (-122.29462 37.80519)
129,OK-N17,322,309,631,Fruitvale BART Station,37.775232,-122.224498,POINT (-122.2245 37.77523)
130,OK-N5,1058,1273,2331,Jack London Square,37.796248,-122.279352,POINT (-122.27935 37.79625)


In [ ]:
df_22 = counts_gdf_22.drop(columns='geometry')
df_22

,id,start_count,end_count,combined_count,name,lat,long
0,BK-A3,1170,1031,2201,College Ave at Alcatraz Ave,37.851376,-122.252523
1,BK-A7,1527,1459,2986,Vine St at Shattuck Ave,37.880222,-122.269592
2,BK-B7,826,736,1562,Virginia St at Shattuck Ave,37.876573,-122.269528
3,BK-C1,966,1310,2276,Fifth St at Delaware St,37.870407,-122.299676
4,BK-C5,1460,1504,2964,North Berkeley BART Station,37.873558,-122.283093
...,...,...,...,...,...,...,...
127,OK-M7,2793,2284,5077,Lake Merritt BART Station,37.797320,-122.265320
128,OK-N1,2981,2987,5968,West Oakland BART Station,37.805189,-122.294617
129,OK-N17,322,309,631,Fruitvale BART Station,37.775232,-122.224498
130,OK-N5,1058,1273,2331,Jack London Square,37.796248,-122.279352


In [ ]:
counts_gdf_23 = create_station_counts_gdf(oak_berk_df_2023)
counts_gdf_23

station names length:  153
station locations length:  160


,id,start_count,end_count,combined_count,name,lat,long,geometry
0,BK-A3,1225,1014,2239,College Ave at Alcatraz Ave,37.851376,-122.252523,POINT (-122.25252 37.85138)
1,BK-A7,1604,1519,3123,Vine St at Shattuck Ave,37.880222,-122.269592,POINT (-122.26959 37.88022)
2,BK-B7,977,989,1966,Virginia St at Shattuck Ave,37.876573,-122.269528,POINT (-122.26953 37.87657)
3,BK-C1,783,1084,1867,Fifth St at Delaware St,37.870407,-122.299676,POINT (-122.29968 37.87041)
4,BK-C5,1394,1405,2799,North Berkeley BART Station,37.873558,-122.283093,POINT (-122.28309 37.87356)
...,...,...,...,...,...,...,...,...
127,OK-M7,3058,2327,5385,Lake Merritt BART Station,37.797320,-122.265320,POINT (-122.26532 37.79732)
128,OK-N1,3270,3274,6544,West Oakland BART Station,37.805189,-122.294617,POINT (-122.29462 37.80519)
129,OK-N17,294,294,588,Fruitvale BART Station,37.775232,-122.224498,POINT (-122.2245 37.77523)
130,OK-N5,946,1337,2283,Jack London Square,37.796248,-122.279352,POINT (-122.27935 37.79625)


In [ ]:
df_23 = counts_gdf_23.drop(columns='geometry')
df_23

,id,start_count,end_count,combined_count,name,lat,long
0,BK-A3,1225,1014,2239,College Ave at Alcatraz Ave,37.851376,-122.252523
1,BK-A7,1604,1519,3123,Vine St at Shattuck Ave,37.880222,-122.269592
2,BK-B7,977,989,1966,Virginia St at Shattuck Ave,37.876573,-122.269528
3,BK-C1,783,1084,1867,Fifth St at Delaware St,37.870407,-122.299676
4,BK-C5,1394,1405,2799,North Berkeley BART Station,37.873558,-122.283093
...,...,...,...,...,...,...,...
127,OK-M7,3058,2327,5385,Lake Merritt BART Station,37.797320,-122.265320
128,OK-N1,3270,3274,6544,West Oakland BART Station,37.805189,-122.294617
129,OK-N17,294,294,588,Fruitvale BART Station,37.775232,-122.224498
130,OK-N5,946,1337,2283,Jack London Square,37.796248,-122.279352


In [ ]:
counts_gdf_24 = create_station_counts_gdf(oak_berk_df_2024)
counts_gdf_24

In [ ]:
df_24 = counts_gdf_24.drop(columns='geometry')
df_24

In [ ]:
counts_gdf_25 = create_station_counts_gdf(oak_berk_df_2025)
counts_gdf_25

In [ ]:
df_25 = counts_gdf_25.drop(columns='geometry')
df_25

In [ ]:
counts_gdf_26 = create_station_counts_gdf(oak_berk_df_2026)
counts_gdf_26

In [ ]:
df_26 = counts_gdf_26.drop(columns='geometry')
df_26

In [ ]:
counts_gdf_20.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_20.geojson"), driver="GeoJSON")

In [ ]:
df_20.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_20.csv"), index=False)

In [ ]:
counts_gdf_21.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_21.geojson"), driver="GeoJSON")

In [ ]:
df_21.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_21.csv"), index=False)

In [ ]:
counts_gdf_22.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_22.geojson"), driver="GeoJSON")

In [ ]:
df_22.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_22.csv"), index=False)

In [ ]:
counts_gdf_23.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_23.geojson"), driver="GeoJSON")

In [ ]:
df_23.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_23.csv"), index=False)

In [ ]:
counts_gdf_24.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_24.geojson"), driver="GeoJSON")

In [ ]:
df_24.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_24.csv"), index=False)

In [ ]:
counts_gdf_25.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_25.geojson"), driver="GeoJSON")

In [ ]:
df_25.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_25.csv"), index=False)

In [ ]:
# counts_gdf_26.to_file(os.path.join(DATA_DIR, "cleaned/stations by year/stations_26.geojson"), driver="GeoJSON")

In [ ]:
# df_26.to_csv(os.path.join(DATA_DIR, "cleaned/stations by year/stations_26.csv"), index=False)